In [33]:
!pip install statsbombpy
from statsbombpy import sb
import pandas as pd
import numpy as np
pd.set_option('future.no_silent_downcasting', True)

In [34]:
import warnings
warnings.filterwarnings('ignore')

In [35]:
competitions_df = sb.competitions()

In [36]:
male_comps = competitions_df[competitions_df['competition_gender'] == 'male']

In [37]:
seasons_list = sorted(male_comps['season_name'].unique())
seasons_list = seasons_list[34:]
seasons_list

['2017/2018',
 '2018',
 '2018/2019',
 '2019/2020',
 '2020',
 '2020/2021',
 '2021/2022',
 '2022',
 '2022/2023',
 '2023',
 '2023/2024',
 '2024']

In [38]:
male_comps = male_comps[male_comps['season_name'].isin(seasons_list)]

In [39]:
match_ids = []

for _, row in male_comps.iterrows():
  comp_id = row['competition_id']
  season_id = row['season_id']

  matches = sb.matches(competition_id=comp_id, season_id=season_id)
  match_ids.extend(matches['match_id'])

In [40]:
len(match_ids)

667

In [41]:
events_list = []
for id in match_ids:
  events_list.append(sb.events(id))
  print(f"Progress: {match_ids.index(id) + 1}/{len(match_ids)}")

Progress: 1/667
Progress: 2/667
Progress: 3/667
Progress: 4/667
Progress: 5/667
Progress: 6/667
Progress: 7/667
Progress: 8/667
Progress: 9/667
Progress: 10/667
Progress: 11/667
Progress: 12/667
Progress: 13/667
Progress: 14/667
Progress: 15/667
Progress: 16/667
Progress: 17/667
Progress: 18/667
Progress: 19/667
Progress: 20/667
Progress: 21/667
Progress: 22/667
Progress: 23/667
Progress: 24/667
Progress: 25/667
Progress: 26/667
Progress: 27/667
Progress: 28/667
Progress: 29/667
Progress: 30/667
Progress: 31/667
Progress: 32/667
Progress: 33/667
Progress: 34/667
Progress: 35/667
Progress: 36/667
Progress: 37/667
Progress: 38/667
Progress: 39/667
Progress: 40/667
Progress: 41/667
Progress: 42/667
Progress: 43/667
Progress: 44/667
Progress: 45/667
Progress: 46/667
Progress: 47/667
Progress: 48/667
Progress: 49/667
Progress: 50/667
Progress: 51/667
Progress: 52/667
Progress: 53/667
Progress: 54/667
Progress: 55/667
Progress: 56/667
Progress: 57/667
Progress: 58/667
Progress: 59/667
Progre

In [43]:
data = pd.concat(events_list)

In [46]:
data.info(verbose=True, show_counts=True)

<class 'pandas.core.frame.DataFrame'>
Index: 2384741 entries, 0 to 2995
Data columns (total 119 columns):
 #    Column                            Non-Null Count    Dtype  
---   ------                            --------------    -----  
 0    50_50                             1967 non-null     object 
 1    ball_receipt_outcome              84654 non-null    object 
 2    ball_recovery_offensive           238 non-null      object 
 3    ball_recovery_recovery_failure    4453 non-null     object 
 4    block_deflection                  839 non-null      object 
 5    block_offensive                   431 non-null      object 
 6    carry_end_location                540005 non-null   object 
 7    clearance_aerial_won              4536 non-null     object 
 8    clearance_body_part               22225 non-null    object 
 9    clearance_head                    11981 non-null    object 
 10   clearance_left_foot               3975 non-null     object 
 11   clearance_right_foot          

In [47]:
xg_features = [31, 36, 35, 71, 72, 74, 79, 80, 77]

In [48]:
xg_feature_labels = []
for i in xg_features:
  xg_feature_labels.append(data.columns[i])
xg_feature_labels

['id',
 'match_id',
 'location',
 'shot_aerial_won',
 'shot_body_part',
 'shot_first_time',
 'shot_technique',
 'shot_type',
 'shot_outcome']

In [49]:
xg_df = data.loc[data['shot_freeze_frame'].notnull(), xg_feature_labels]
xg_df = xg_df.fillna(False)

In [50]:
xg_df[['shot_location_x', 'shot_location_y']] = xg_df['location'].tolist()

In [51]:
cols = list(xg_df.columns)
sloc_idx = cols.index('location')
cols.remove('shot_location_x')
cols.remove('shot_location_y')
cols.remove('location')
cols[sloc_idx:sloc_idx] = ['shot_location_x', 'shot_location_y']

In [52]:
xg_df = xg_df[cols]

In [53]:
xg_df.reset_index(drop=True, inplace=True)

In [54]:
xg_df

,id,match_id,shot_location_x,shot_location_y,shot_aerial_won,shot_body_part,shot_first_time,shot_technique,shot_type,shot_outcome
0,452ac1fc-6e97-459f-8819-62bc82078292,3895292,107.0,53.4,False,Right Foot,False,Half Volley,Open Play,Off T
1,345c34d9-2c2f-4cec-860e-3b3704ebc4f3,3895292,96.5,39.9,False,Right Foot,True,Normal,Open Play,Blocked
2,6627cd42-a35b-4e4e-9fd3-05d3269bca36,3895292,88.2,45.5,False,Right Foot,False,Normal,Open Play,Blocked
3,3d1c53e1-4b07-4648-806c-2dc0b2ee5347,3895292,98.5,49.2,False,Left Foot,False,Normal,Free Kick,Saved
4,bcdb8320-5e4f-4c64-910a-f4ff7ace5e8b,3895292,111.2,37.8,True,Head,False,Normal,Open Play,Saved
...,...,...,...,...,...,...,...,...,...,...
16646,ff89d5ba-8010-4f23-9e20-176ac0452f78,3788744,95.5,31.6,False,Left Foot,True,Normal,Open Play,Off T
16647,57a168e4-ab59-40e3-83ce-caeb23077097,3788744,99.4,48.8,False,Right Foot,True,Normal,Open Play,Blocked
16648,8617911a-b341-4349-869d-f2355a512524,3788744,94.9,40.8,False,Right Foot,True,Half Volley,Open Play,Blocked
16649,829a59d9-68b6-4608-9dd9-d879fb0920c5,3788744,104.5,32.4,False,Right Foot,True,Half Volley,Open Play,Blocked


In [55]:
# xg_df.to_csv('xg_df.csv', index=False)